# 04 - Layer C: Inventory-Routing Problem (IRP)

Decomposition: Layer A/B `(s,S)` → daily delivery requests → Capacitated VRP (OR-Tools) → realized transport cost.

Coordinates are sampled from the logistics dataset (no shared key with Kaggle store IDs — same caveat as lead time). Vehicle capacities are parsed from real Vehicle Type strings (`14MT`, `35MT`, …).

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
from src.layer_c_routing import (
    haversine_distance_km, parse_vehicle_capacity_tonnes, get_store_coordinates,
    build_delivery_requests, build_vrp_model,
)

## Why decomposition, not joint optimization

Full IRP (simultaneously optimizing inventory + routes every day) is NP-hard at a scale beyond exact solution for 10 stores x 50 items x many days. Decomposition: solve Layer A/B policy first (already done), convert reorder triggers into a daily delivery-request list, then solve routing as a Capacitated VRP per day. Feed the realized transportation cost back into Layer A's fixed order cost `K`, re-solve, repeat (2-3 iterations typically stabilizes).

## Step 1 (ready) - Extract real coordinates from the logistics dataset

In [2]:
coords = get_store_coordinates()
coords.head(10)

2026-08-28 22:00:23 | INFO     | src.layer_c_routing | Loaded 353 distinct destination coordinates from logistics dataset


,location_name,latitude,longitude
0,"Pondur, Kanchipuram, Tamil Nadu",12.930429,79.931163
1,"Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000
4,"Singaperumalkoil, Kanchipuram, Tamil Nadu",12.786517,79.975221
5,"Mookandapalli, Krishnagiri, Tamil Nadu",12.746894,77.806168
6,"Heromotocorp Ltd,Alwar,Rajasthan",28.207000,76.856000
9,"Shive, Pune, Maharashtra",18.750621,73.877190
11,"Ashok Leyland Plant 2-Hosur,Hosur,Karnataka",12.766000,77.786000
16,"Tractors & Farms Equipments Limited,Kanchipura...",12.778000,80.025000
22,"Lucas Tvs Ltd-Ambattur,Chennai,Tamil Nadu",13.102000,80.194000
23,"Lucas Tvs Ltd-Pondy,Pondy,Pondicherry",11.872000,79.632000


## Step 1b (ready) - Vehicle capacity parsing from free-text Vehicle Type

In [3]:
import pandas as pd

df = pd.read_excel('../data/raw/Transportation__Logistics_Tracking_Dataset.xlsx',
                    sheet_name='Primary Data')
sample_types = df['Vehicle Type'].dropna().unique()[:15]
for vt in sample_types:
    print(f'{vt!r:55s} -> {parse_vehicle_capacity_tonnes(vt)} MT')

'32 FT Multi-Axle 14MT - HCV'                           -> 14.0 MT
'32 FT Single-Axle 7MT - HCV'                           -> 7.0 MT
'19 FT OPEN BODY 8 MT'                                  -> 8.0 MT
'20 FT SXL Container'                                   -> None MT
'32 FT Multi-Axle MXL 18MT'                             -> 18.0 MT
'20 FT CLOSE 7MT-MCV '                                  -> 7.0 MT
'1 MT Tata Ace (Open Body)'                             -> 1.0 MT
'24 FT SXL Container'                                   -> None MT
'1 MT Tata Ace (Closed Body)'                           -> 1.0 MT
'1.5 MT Pickup (Open Body)'                             -> 1.5 MT
'14 FT Open - 3 MT'                                     -> 3.0 MT
'40 FT 3XL Trailer 35MT'                                -> 35.0 MT
'1.5 MT Vehicle (Closed Body)'                          -> 1.5 MT
'17 FT Container'                                       -> None MT
'24 / 26 FT Taurus Open 21MT - HCV'                     -> 21.0 MT


## Step 2 (ready) - Distance calculation example

In [4]:
# Example: distance between the first two distinct coordinates
a, b = coords.iloc[0], coords.iloc[1]
dist = haversine_distance_km(a['latitude'], a['longitude'], b['latitude'], b['longitude'])
print(f"{a['location_name']} -> {b['location_name']}: {dist:.1f} km")

Pondur, Kanchipuram, Tamil Nadu -> Daimler India Commercial Vehicles,Kanchipuram,Tamil Nadu: 10.5 km


## Step 3 - Build delivery requests

Simulate an ERP on-hand snapshot: start near each pair's reorder point `s`. Any pair with `on_hand <= s` requests `S - on_hand` units.

In [5]:
policies = pd.read_csv('../data/processed/layer_a_policies.csv')
rng = np.random.default_rng(42)
current_inventory = policies[['store', 'item']].copy()
current_inventory['on_hand'] = (policies['s'] * rng.uniform(0.2, 1.15, len(policies))).astype(int)

day = pd.Timestamp('2017-12-31')
delivery_requests = build_delivery_requests(policies, current_inventory, day)
print(f'{len(delivery_requests)} requests across {delivery_requests["store"].nunique()} stores')
delivery_requests

2026-08-28 22:00:24 | INFO     | src.layer_c_routing | Delivery requests for 2017-12-31: 9 SKUs across 9 stores (from 10 inventory rows)


9 requests across 9 stores


,store,item,quantity,requested_date
0,1,10,1047,2017-12-31
1,2,24,1217,2017-12-31
2,3,5,579,2017-12-31
3,4,6,1012,2017-12-31
4,5,12,977,2017-12-31
5,7,17,609,2017-12-31
6,8,12,1189,2017-12-31
7,9,7,1056,2017-12-31
8,10,1,643,2017-12-31


## Step 4 - Solve the Capacitated VRP (OR-Tools)

Assign 10 real destination coordinates to stores 1–10 (modeling choice, seed=42). Depot is a real origin from the logistics file. Vehicle capacities come from parsed `Vehicle Type` tonnage.

In [6]:
from collections import Counter

pool = get_store_coordinates()
picked = pool.sample(n=10, random_state=42).reset_index(drop=True)
store_coordinates = picked.copy()
store_coordinates.insert(0, 'store', range(1, 11))

logistics = pd.read_excel('../data/raw/Transportation__Logistics_Tracking_Dataset.xlsx',
                          sheet_name='Primary Data')
origin = logistics[['Origin Location Latitude', 'Origin Location Longitude']].dropna().iloc[0]
depot = (float(origin.iloc[0]), float(origin.iloc[1]))

parsed = [parse_vehicle_capacity_tonnes(vt) for vt in logistics['Vehicle Type'].dropna()]
vehicle_capacities = [k for k, _ in Counter(c for c in parsed if c).most_common(6)]
print('depot', depot)
print('vehicle_capacities (MT)', vehicle_capacities)

result = build_vrp_model(delivery_requests, store_coordinates, depot, vehicle_capacities)
print(
    f"status={result['status']}  vehicles={result['n_vehicles_used']}  "
    f"{result['total_distance_km']} km  cost={result['total_cost']}"
)
print()
for r in result['routes']:
    print(f"vehicle {r['vehicle']}: {r['distance_km']} km, load {r['load_kg']} kg / {r['capacity_kg']} kg")
    print('   ' + ' -> '.join(r['stops']))
result['stops']

2026-08-28 22:00:26 | INFO     | src.layer_c_routing | Loaded 353 distinct destination coordinates from logistics dataset


depot (18.750621, 73.87719)
vehicle_capacities (MT) [14.0, 35.0, 7.0, 27.0, 18.0, 21.0]
2026-08-28 22:00:27 | INFO     | src.layer_c_routing | Solving CVRP: 9 stops, 9 vehicles, total demand 83.3 t


2026-08-28 22:00:35 | INFO     | src.layer_c_routing | VRP solved: 3 routes, 6699.3 km, cost 10198.94


status=SOLVED  vehicles=3  6699.29 km  cost=10198.94

vehicle 6: 2077.37 km, load 28370 kg / 35000 kg
   depot -> store 2 -> store 10 -> store 5 -> depot
vehicle 7: 3557.36 km, load 28240 kg / 35000 kg
   depot -> store 9 -> store 3 -> store 8 -> depot
vehicle 8: 1064.56 km, load 26680 kg / 35000 kg
   depot -> store 7 -> store 1 -> store 4 -> depot


,vehicle,seq,stop,demand_kg,cumul_load_kg
0,6,0,depot,0,0
1,6,1,store 2,12170,12170
2,6,2,store 10,6430,18600
3,6,3,store 5,9770,28370
4,6,4,depot,0,28370
5,7,0,depot,0,0
6,7,1,store 9,10560,10560
7,7,2,store 3,5790,16350
8,7,3,store 8,11890,28240
9,7,4,depot,0,28240
